In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [24]:
from google.colab import files
uploaded = files.upload()

Saving NaiveBayesWorkflow.jpg to NaiveBayesWorkflow.jpg


In [25]:
from IPython.display import Image, display
filename = list(uploaded.keys())[0]
print(filename)
display(Image(filename))

Output hidden; open in https://colab.research.google.com to view.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Naive_Bayes_Training_Dataset.xlsx to Naive_Bayes_Training_Dataset.xlsx


In [7]:
df = pd.read_excel("Naive_Bayes_Training_Dataset.xlsx")
print(df.shape)
df.head(20)

(20, 5)


,day,season,wind,rain,class
0,weekday,spring,none,none,on time
1,weekday,winter,none,slight,on time
2,weekday,winter,none,slight,on time
3,weekday,winter,high,heavy,late
4,saturday,summer,normal,none,on time
5,weekday,autumn,normal,none,very late
6,holiday,summer,high,slight,on time
7,sunday,summer,normal,none,on time
8,weekday,winter,high,heavy,very late
9,weekday,summer,none,slight,on time


In [8]:
target = df.columns[-1]
features = [c for c in df.columns if c != target]

print("Target:", target)
print("Features:", features)

priors = df[target].value_counts(normalize=True).to_dict()
print("Priors:", priors)

Target: class
Features: ['day', 'season', 'wind', 'rain']
Priors: {'on time': 0.7, 'very late': 0.15, 'late': 0.1, 'cancelled': 0.05}


In [9]:
def likelihood(feature, value, cls, alpha=0):
    subset = df[df[target] == cls]
    count = (subset[feature] == value).sum()
    n_values = df[feature].nunique()
    return (count + alpha) / (len(subset) + alpha * n_values)

In [16]:
def predict(instance, alpha=0):
    scores = {}
    for cls in priors:
        score = priors[cls]
        for feature in features:
            score *= likelihood(feature, instance[feature], cls, alpha)
        scores[cls] = score

    total = sum(scores.values())
    probs = {c: (s / total if total else 0) for c, s in scores.items()}
    return max(scores, key=scores.get), scores, probs

In [19]:
print(df.columns.tolist())
for c in features:
    print(c, "->", df[c].unique())

['day', 'season', 'wind', 'rain', 'class']
day -> ['weekday' 'saturday' 'holiday' 'sunday']
season -> ['spring' 'winter' 'summer' 'autumn']
wind -> ['none' 'high' 'normal']
rain -> ['none' 'slight' 'heavy']


In [20]:
values = ["weekday", "winter", "high", "heavy"]   # in the same order as your feature columns
new_row = dict(zip(features, values))
print(new_row)

prediction, scores, probs = predict(new_row)
print("Raw scores:", scores)
print("Normalized:", probs)
print("Prediction:", prediction)

{'day': 'weekday', 'season': 'winter', 'wind': 'high', 'rain': 'heavy'}
Raw scores: {'on time': np.float64(0.0013119533527696792), 'very late': np.float64(0.022222222222222216), 'late': np.float64(0.0125), 'cancelled': np.float64(0.0)}
Normalized: {'on time': np.float64(0.03640858523429599), 'very late': np.float64(0.6166985054500504), 'late': np.float64(0.34689290931565353), 'cancelled': np.float64(0.0)}
Prediction: very late


In [21]:
classes = list(df[target].unique())
class_counts = df[target].value_counts()

rows = {}
for f in features:
    for v in df[f].unique():
        row = {}
        for c in classes:
            n = ((df[f] == v) & (df[target] == c)).sum()
            d = class_counts[c]
            row[f"Class = {c}"] = f"{n}/{d} = {n/d:.2f}"
        rows[f"{f} = {v}"] = row


total = len(df)
rows["prior probability"] = {
    f"Class = {c}": f"{class_counts[c]}/{total} = {class_counts[c]/total:.2f}"
    for c in classes
}

table = pd.DataFrame(rows).T
table

,Class = on time,Class = late,Class = very late,Class = cancelled
day = weekday,9/14 = 0.64,1/2 = 0.50,3/3 = 1.00,0/1 = 0.00
day = saturday,2/14 = 0.14,1/2 = 0.50,0/3 = 0.00,1/1 = 1.00
day = holiday,2/14 = 0.14,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
day = sunday,1/14 = 0.07,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
season = spring,4/14 = 0.29,0/2 = 0.00,0/3 = 0.00,1/1 = 1.00
season = winter,2/14 = 0.14,2/2 = 1.00,2/3 = 0.67,0/1 = 0.00
season = summer,6/14 = 0.43,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
season = autumn,2/14 = 0.14,0/2 = 0.00,1/3 = 0.33,0/1 = 0.00
wind = none,5/14 = 0.36,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
wind = high,4/14 = 0.29,1/2 = 0.50,1/3 = 0.33,1/1 = 1.00


In [22]:
order = {
    "day": ["weekday", "saturday", "sunday", "holiday"],
    "season": ["spring", "summer", "autumn", "winter"],
    "wind": ["none", "high", "normal"],
    "rain": ["none", "slight", "heavy"],
}
classes = ["on time", "late", "very late", "cancelled"]

rows = {}
for f in features:
    for v in order[f]:
        row = {}
        for c in classes:
            n = ((df[f] == v) & (df[target] == c)).sum()
            d = class_counts[c]
            row[f"Class = {c}"] = f"{n}/{d} = {n/d:.2f}"
        rows[f"{f} = {v}"] = row

rows["prior probability"] = {
    f"Class = {c}": f"{class_counts[c]}/{total} = {class_counts[c]/total:.2f}"
    for c in classes
}

table = pd.DataFrame(rows).T
table

,Class = on time,Class = late,Class = very late,Class = cancelled
day = weekday,9/14 = 0.64,1/2 = 0.50,3/3 = 1.00,0/1 = 0.00
day = saturday,2/14 = 0.14,1/2 = 0.50,0/3 = 0.00,1/1 = 1.00
day = sunday,1/14 = 0.07,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
day = holiday,2/14 = 0.14,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
season = spring,4/14 = 0.29,0/2 = 0.00,0/3 = 0.00,1/1 = 1.00
season = summer,6/14 = 0.43,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
season = autumn,2/14 = 0.14,0/2 = 0.00,1/3 = 0.33,0/1 = 0.00
season = winter,2/14 = 0.14,2/2 = 1.00,2/3 = 0.67,0/1 = 0.00
wind = none,5/14 = 0.36,0/2 = 0.00,0/3 = 0.00,0/1 = 0.00
wind = high,4/14 = 0.29,1/2 = 0.50,1/3 = 0.33,1/1 = 1.00


In [23]:
new_row = dict(zip(features, ["weekday", "winter", "high", "heavy"]))
prediction, scores, probs = predict(new_row)
print(scores)
print(probs)
print("Prediction:", prediction)

{'on time': np.float64(0.0013119533527696792), 'very late': np.float64(0.022222222222222216), 'late': np.float64(0.0125), 'cancelled': np.float64(0.0)}
{'on time': np.float64(0.03640858523429599), 'very late': np.float64(0.6166985054500504), 'late': np.float64(0.34689290931565353), 'cancelled': np.float64(0.0)}
Prediction: very late
